# Görüntü İşleme – Ara Sınav Projesi
## Klasik yöntemlerle Fluo-N2DH-GOWT1 hücre sayımı

**Yusuf Ekrem Ünlü – 254329042**  ·  seed = 29042 + 2026 = **31068**

Bu notebook tüm deneyleri `src/` altındaki modülleri çağırarak sırayla çalıştırır.
Veri seti klasörü (`01`, `02`, `01_GT`, `02_GT` içeren klasör) şu konumlardan birinde aranır:
`data/Fluo-N2DH-GOWT1`, `../Fluo-N2DH-GOWT1/Fluo-N2DH-GOWT1`, `../Fluo-N2DH-GOWT1` veya
`GOWT1_DIR` ortam değişkeni.

### 0. Ortam (yalnızca Colab'da gerekli)

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # repo'yu kendi GitHub adresinizle klonlayın
    # !git clone https://github.com/<kullanici>/<repo>.git && %cd <repo>
    if not os.path.exists('data/Fluo-N2DH-GOWT1'):
        !mkdir -p data && cd data && wget -q https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip && unzip -q Fluo-N2DH-GOWT1.zip
    !pip -q install -r requirements.txt

In [ ]:
import sys, json
sys.path.insert(0, 'src')
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.max_colwidth', 60)
from config import SEED, find_data_dir
print('seed =', SEED, '| veri klasörü =', find_data_dir())

### 1. Tekrarlanabilir kare seçimi (Bölüm 3)

In [ ]:
import select_frames
sel = select_frames.main()
sel

### 2. Veri inceleme ve referans sayımlar (Bölüm 2 ve 4)

In [ ]:
from data_io import reference_count
sel_dv = sel[sel.split != 'test'].copy()   # test referansları bu aşamada OKUNMAZ
sel_dv['reference'] = [reference_count(s, f) for s, f in zip(sel_dv.sequence, sel_dv.frame)]
sel_dv

### 3–4. Baseline ve kontrollü deneyler (Bölüm 5–10)

In [ ]:
import experiments
exp = experiments.main()
exp[['exp_id','stage','note','intensity','filter','segmentation','morphology','split','dev_MAE','dev_RMSE','dev_Bias','mean_time_s']]

### 5. Doğrulama ve nihai pipeline seçimi (Bölüm 11)

In [ ]:
import validate
val, final = validate.main()
val

### 6. KİLİTLİ test tahmini (Bölüm 12)
Bu hücre test referanslarını okumaz. Çıktı `test_predictions_locked.csv`'ye yazılır.
**Bu noktada GitHub'a ayrı bir commit atılmalıdır** (README'deki adımlar).

In [ ]:
import predict_test
predict_test.main()

### 7. Test değerlendirmesi ve zamansal analiz (Bölüm 13–14)

In [ ]:
import evaluate_test
res, summ = evaluate_test.main()
display(res); display(summ)

### 8. Şekiller

In [ ]:
import figures
figures.main()
for f in ['fig1_flowchart.png', 'fig_params.png', 'fig5_temporal.png', 'fig0_dataset_histograms.png']:
    display(Image(filename='results/figures/' + f))

Veri setine ait görüntü içeren şekiller (`results/figures_private/`) GitHub'a yüklenmez;
yerelde görmek için aşağıdaki hücreyi çalıştırın. GitHub'a göndermeden önce notebook çıktılarını temizleyin.

In [ ]:
# for f in ['fig2_intensity.png','fig3_seg_morph.png','fig_filters.png','fig4_stages.png','fig6_examples.png']:
#     display(Image(filename='results/figures_private/' + f))